In [1]:
import glob
import os
import shutil
from subprocess import call

import xarray as xa
import numpy as np
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
from cartopy.feature import LAND, COASTLINE
import cmocean.cm as ocm

In [2]:
proj = ccrs.NorthPolarStereo(true_scale_latitude=90, central_longitude=-45)

In [ ]:
idir = '/data1/antonk/nextsim_reanalysis/2023'
odir = '/data1/antonk/nextsim_reanalysis/2023/pngs'
ifiles = sorted(glob.glob(f'{idir}/[0,1]?/*nc'))
len(ifiles)

In [4]:
titles = {
    'sithick': 'Thickness',
    'siage': 'Ice Age',
    'siconc_my': 'MYI concentration',
    'si_ridge_ratio': 'Ridged ice (roughness)',
}

cmaps = {
    'sithick': 'viridis',
    'siage': 'jet',
    'siconc_my': ocm.ice,
    'si_ridge_ratio': 'plasma',
}

clims = {
    'sithick': [0, 3],
    'siage': [0, 3],
    'siconc_my': [0, 1],
    'si_ridge_ratio': [0, 1],
}

In [ ]:
step = 2
for ifile in ifiles[::step]:
    datestr = os.path.basename(ifile).split('_')[0]
    ofile0 = f'{odir}/{list(titles.keys())[0]}_{datestr}.png'
    if os.path.exists(ofile0):
        print('Skip', ifile)
        continue
    try:
        with xa.open_dataset(ifile) as ds:
            data = {name: ds[name][0].to_numpy() for name in titles}
            x = ds.x.to_numpy()
            y = ds.y.to_numpy()
    except:
        print('Canot open ', ifile)
        continue
    for name in titles:
        ofile = f'{odir}/{name}_{datestr}.png'
        print(ofile)
        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(1, 1, 1, projection=proj)
        im = ax.imshow(
            data[name],
            interpolation='nearest',
            cmap=cmaps[name],
            clim=clims[name],
            extent=[x[0], x[-1], y[0], y[-1]],
            origin='lower'
        )
        ax.add_feature(LAND)
        ax.add_feature(COASTLINE)
        ax.set_xlim([-2.5e6, 1.5e6])
        ax.set_ylim([-2e6, 2e6])
        ax.set_title(f"{titles[name]} {os.path.basename(ifile).split('_')[0]}")
        plt.colorbar(im, ax=ax, shrink=0.5)
        plt.savefig(ofile, dpi=100, bbox_inches='tight', pad_inches=0.1)
        plt.close()


In [ ]:
for name in titles:
    png_files = sorted(glob.glob(f'{odir}/{name}*png'))
    frame_files = sorted(glob.glob(f'{odir}/frames/*png'))
    [os.remove(f) for f in frame_files]
    for i, png_file in enumerate(png_files):
        frame_name = f'{odir}/frames/frame_{i:05}.png'
        shutil.copy(png_file, frame_name)
    cmd = f"ffmpeg -y -r 6 -i {odir}/frames/frame_%05d.png -s 728x662 -r 25 -c:v libx264 -crf 20  -pix_fmt yuv420p ~/{name}.mov"
    call(cmd, shell=True)